# 1. Döntési fa modell

## Modell relevanciája

A döntési fa (Decision Tree Classifier) egy felügyelt tanulási algoritmus, amely klasszifikációs problémák megoldására használható. A modell hierarchikus döntési szabályok alapján választja szét az osztályokat.

A prediktív karbantartási feladatban a cél annak előrejelzése, hogy egy adott gépállapothoz tartozik-e meghibásodás:

- 0 = nincs meghibásodás
- 1 = meghibásodás

A döntési fa előnye, hogy:

- jól interpretálható,
- képes nemlineáris kapcsolatok kezelésére,
- nem érzékeny a változók skálájára,
- automatikusan képes fontos változókat kiválasztani.

A logisztikus regresszióval szemben a döntési fa nem lineáris modellt épít, hanem egymás utáni döntési szabályokat hoz létre.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)


## 2. Train és test csv beolvasás

In [ ]:
train_df = pd.read_csv("../../DataCleaning/train.csv")
test_df = pd.read_csv("../../DataCleaning/test.csv")

## 3. X és y szétválasztása

- `X_train`: tanító magyarázó változók
- `y_train`: tanító célváltozó
- `X_test`: teszt magyarázó változók
- `y_test`: teszt célváltozó

In [ ]:
target_col = "target"

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## 4. Osztályeloszlás ellenőrzése

In [ ]:
def show_class_distribution(y, dataset_name):
    counts = y.value_counts().sort_index()
    ratios = y.value_counts(normalize=True).sort_index() * 100

    dist_df = pd.DataFrame({
        "Count": counts,
        "Ratio (%)": ratios.round(2)
    })

    print(f"\n{dataset_name} osztályeloszlás:")
    display(dist_df)

    return dist_df

train_dist = show_class_distribution(y_train, "Train")
test_dist = show_class_distribution(y_test, "Test")

## 5. Modellspecifikus előfeldolgozás

A döntési fa nem érzékeny a változók skálájára, ezért nincs szükség standardizálásra.

In [ ]:
numeric_columns = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

binary_features = ["type_L", "type_M"]

numeric_features = [
    col for col in numeric_columns
    if col not in binary_features
]

selected_features = numeric_features + binary_features

print("Numerikus változók száma:", len(numeric_features))
print("Összes felhasznált feature:", len(selected_features))

## 6. Pipeline létrehozása

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("features", "passthrough", selected_features)
    ],
    remainder="drop"
)

decision_tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            random_state=42,
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            class_weight="balanced"
        )
    )
])

decision_tree_pipeline

## 7. Keresztvalidáció beállítása

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv

## 8. Modell értékelése keresztvalidációval

In [ ]:
cv_results = cross_validate(
    estimator=decision_tree_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True
)

cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Train mean": [
        cv_results["train_accuracy"].mean(),
        cv_results["train_precision"].mean(),
        cv_results["train_recall"].mean(),
        cv_results["train_f1"].mean(),
        cv_results["train_roc_auc"].mean()
    ],
    "Validation mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ]
})

cv_summary = cv_summary.round(4)

cv_summary

## 9. Hiperparaméter hangolás GridSearchCV segítségével

In [ ]:
param_grid = {
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 3, 5],
    "model__criterion": ["gini", "entropy"]
}

grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

## 10. Legjobb hiperparaméterek

In [ ]:
print("Legjobb paraméterek:")
print(grid_search.best_params_)

print("\nLegjobb CV F1-score:")
print(round(grid_search.best_score_, 4))

## 11. Végső modell kiválasztása

In [ ]:
final_decision_tree_model = grid_search.best_estimator_

final_decision_tree_model

## 12. Predikció a teszthalmazon

In [ ]:
y_pred = final_decision_tree_model.predict(X_test)

y_prob = final_decision_tree_model.predict_proba(X_test)[:, 1]

## 13. Modell kiértékelése

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

results_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results_df = results_df.round(4)

results_df

## 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm
).plot(ax=ax)

plt.title("Decision Tree - Confusion Matrix")
plt.show()

## 15. Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

## 16. ROC görbe

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)

plt.title("Decision Tree - ROC Curve")
plt.show()

## 17. Feature importance

In [ ]:
model = final_decision_tree_model.named_steps["model"]

feature_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

feature_importance_df.head(15)

## 18. Feature importance vizualizáció

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance_df["Feature"][:10][::-1],
    feature_importance_df["Importance"][:10][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importance - Decision Tree")

plt.show()

## 19. Döntési fa vizualizáció

In [ ]:
model = final_decision_tree_model.named_steps["model"]

plt.figure(figsize=(20, 10))

plot_tree(
    model,
    feature_names=selected_features,
    class_names=["No Failure", "Failure"],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)

plt.title("Decision Tree Visualization")
plt.show()

## 20. Eredmények értelmezése

A döntési fa modell jól interpretálható szabályalapú klasszifikációt biztosít.

A feature importance elemzés alapján meghatározható, mely változók járulnak hozzá leginkább a meghibásodások előrejelzéséhez.

In [ ]:
print("A döntési fa modell sikeresen lefutott.")

## 20. Threshold összehasonlítás

A különböző threshold értékek eltérő precision és recall eredményeket adhatnak.

Ez különösen fontos imbalance classification problémáknál.


In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:

    y_threshold_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_threshold_pred),
        "Precision": precision_score(y_test, y_threshold_pred),
        "Recall": recall_score(y_test, y_threshold_pred),
        "F1-score": f1_score(y_test, y_threshold_pred)
    })

threshold_comparison_df = pd.DataFrame(threshold_results)

threshold_comparison_df = threshold_comparison_df.round(4)

threshold_comparison_df

## 21. Threshold eredmények exportálása

In [ ]:
threshold_comparison_df.to_csv(
    "decision_tree_threshold_comparison.csv",
    index=False
)

print("Threshold comparison export kész.")

## 22. Final model eredmények exportálása

In [ ]:
results_df.to_csv(
    "decision_tree_final_results.csv",
    index=False
)

feature_importance_df.to_csv(
    "decision_tree_feature_importance.csv",
    index=False
)

print("Final results export kész.")

## 23. Összes fontos output exportálása CSV-be

A notebook automatikusan elmenti a legfontosabb eredményeket CSV fájlokba.


In [ ]:
# Cross-validation summary export
cv_summary.to_csv(
    "decision_tree_cv_summary.csv",
    index=False
)

# Final evaluation metrics export
results_df.to_csv(
    "decision_tree_final_results.csv",
    index=False
)

# Threshold comparison export
threshold_comparison_df.to_csv(
    "decision_tree_threshold_comparison.csv",
    index=False
)

# Feature importance export
feature_importance_df.to_csv(
    "decision_tree_feature_importance.csv",
    index=False
)

# Prediction export
prediction_export_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability": y_prob
})

prediction_export_df.to_csv(
    "decision_tree_predictions.csv",
    index=False
)

print("Minden CSV export sikeresen elkészült.")
